In [ ]:
import pandas as pd

# 1. CLEAN master_socioeconomic_panel.csv
# This fixes the name mismatch for DC and removes hidden spaces
socio_df = pd.read_csv('master_socioeconomic_panel.csv')

# Define the long name found in the Census data
dc_long_name = 'District of Columbia (Note: Cities must be listed as Full Names.  Abbrevations are not supported for Cities)'

socio_df['State'] = socio_df['State'].replace({dc_long_name: 'DC'}).str.strip()


# 2. CLEAN Ads_final.xlsx - Ads_final.csv
# This handles targeting nulls and adds a Year column for the merge
ads_df = pd.read_excel('/content/Ads_final.xlsx')

# Clean state codes and fill targeting gaps
ads_df['state_code'] = ads_df['state_code'].str.strip().str.upper()
ads_df['age_targeting'] = ads_df['age_targeting'].fillna('Broad Targeting')
ads_df['gender_targeting'] = ads_df['gender_targeting'].fillna('Broad Targeting')
# Create the "Bridge" Key directly from the date + state
# This creates values like "FL2024" or "NY2020"
ads_df['State_Year_Key'] = ads_df['state_code'].astype(str) + ads_df['date_range_start'].dt.year.astype(str)
socio_df['State_Year_Key'] = socio_df['State'].astype(str) + socio_df['Year'].astype(str)

# Move the Key to the very first column
ads_cols = ['State_Year_Key'] + [c for c in ads_df.columns if c != 'State_Year_Key']
ads_df = ads_df[ads_cols]

socio_cols = ['State_Year_Key'] + [c for c in socio_df.columns if c != 'State_Year_Key']
socio_df = socio_df[socio_cols]

# Save back to your original filenames
ads_df.to_csv('Ads_final.csv', index=False)
socio_df.to_csv('master_socioeconomic_panel.csv', index=False)

print("Success: Files saved with State_Year_Key as the first column. No separate Year column added to Ads.")

Success: Files saved with State_Year_Key as the first column. No separate Year column added to Ads.


In [ ]:
# --- THE AUDIT CODE ---

# 1. Create the Year column (This is required for a valid merge)
ads_df['date_range_start'] = pd.to_datetime(ads_df['date_range_start'])
ads_df['Year'] = ads_df['date_range_start'].dt.year

# 2. Perform a test merge
# We use 'left' join so we can see which ads FAILED to find socioeconomic data
merged_test = pd.merge(ads_df, socio_df, left_on=['state_code', 'Year'], right_on=['State', 'Year'], how='left')

# 3. IDENTIFY THE CULPRITS
# This filters for rows where the merge failed (e.g., Median_Income is NaN)
failed_matches = merged_test[merged_test['Median_Income'].isnull()]

print("--- MERGE AUDIT REPORT ---")
if failed_matches.empty:
    print("Success! No null values found in the merge.")
else:
    print(f"Total rows with Nulls: {len(failed_matches)}")
    print("\nStates causing Nulls (Top 5):")
    print(failed_matches['state_code'].value_counts().head(5))

    print("\nYears causing Nulls:")
    print(failed_matches['Year'].value_counts())

    # Check if 'DC' specifically failed
    if 'DC' in failed_matches['state_code'].values:
        print("\nALERT: 'DC' is still causing nulls. The name replacement did not work.")

Success: Population data added to the socioeconomic panel.
